# L10 · 概率与统计：在不确定中做决策

**学习目标**
- 理解「概率」「均值/方差」「正态分布」
- 会用模拟估计概率（蒙特卡洛）
- 体会 AI 如何在「不确定」里做最优决策

**前置依赖**：L07 numpy、L08 pandas、L09 可视化  
**预计时长**：45 分钟  
**技术栈**：`numpy`、`matplotlib`

---

## 概念讲解：世界充满随机，概率是你唯一靠谱的指南针

明天下雨吗？这个用户会买单吗？模型输出靠谱吗？——全是不确定。
**概率** 把「不确定」变成可计算的「可能性」。**统计** 从有限样本推断整体规律。
AI 的本质：在概率世界里，选期望收益最高的动作。

## 第一步：用模拟估计概率

In [ ]:
import numpy as np

# 问题：掷两颗骰子，点数和 ≥ 8 的概率有多大？
trials = 200_000
rolls = np.random.randint(1, 7, size=(trials, 2))
p = (rolls.sum(axis=1) >= 8).mean()
print(f"模拟 {trials:,} 次，P(和≥8) ≈ {p:.4f}（理论值 15/36≈0.4167）")

## 第二步：正态分布 —— 自然界的「钟形」

In [ ]:
import matplotlib.pyplot as plt
x = np.random.normal(170, 8, 10000)   # 均值170，标准差8（身高cm）
print(f"样本均值 {x.mean():.1f}，标准差 {x.std():.1f}")
plt.hist(x, bins=40, color="#1f77b4", alpha=0.8)
plt.title("身高分布（正态分布）"); plt.xlabel("cm")
plt.show()

## 第三步：用概率做「最优决策」

In [ ]:
# 两家广告投放：A 平均转化 5%、波动小；B 平均 6%、波动大
A = np.random.normal(0.05, 0.005, 10000)
B = np.random.normal(0.06, 0.02, 10000)
print("A 平均转化：", round(A.mean(), 4), " B 平均转化：", round(B.mean(), 4))
print("B 比 A 高的概率：", round((B > A).mean(), 3), "（别被平均值骗了，要看分布）")

# 🎯 AHA 顿悟单元格：蒙特卡洛赌场 · 看见「庄家优势」

运行下面代码。计算机会**模拟 10 万人玩「猜大小」游戏**，并画出：
你的资金随局数变化的曲线 + 最终盈亏分布。你会「看见」为什么长期必输——
**概率站在庄家那边**。这就是 AI 风控/决策的核心直觉。

> 一个能模拟 10 万次博弈并量化「期望收益」的程序，正是量化金融和强化学习做决策的同一种思维。

In [ ]:
# ===== 运行我！看 10 万局博弈后资金曲线 =====
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(3)

players = 200
rounds = 500
# 每局：赢+1（概率0.48，庄家占优），输-1
wins = np.random.rand(players, rounds) < 0.48
steps = np.where(wins, 1, -1)
cumulative = steps.cumsum(axis=1) + 100   # 起始资金 100

plt.figure(figsize=(9, 4))
for i in range(min(players, 30)):
    plt.plot(cumulative[i], color="#888", alpha=0.3)
plt.plot(cumulative.mean(axis=0), color="#d62728", lw=2.5, label="平均资金")
plt.axhline(100, color="#2ca02c", ls="--", label="起始100")
plt.title(f"{players} 人 × {rounds} 局博弈（赢率仅48%，庄家优势）")
plt.xlabel("局数"); plt.ylabel("资金"); plt.legend()
plt.show()
print(f"  💸 平均玩家 {rounds} 局后资金：{cumulative[:, -1].mean():.1f}（低于100=长期必输）")
print(f"  📉 破产(<0)比例：{(cumulative[:, -1] <= 0).mean()*100:.1f}%")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：均值陷阱（B 平均高但方差大）；蒙特卡洛估计思想。  
**易错点**：`axis` 参数；`cumsum` 维度理解。  
**AHA 机制**：10 万局博弈可视化，直观呈现「概率劣势→必输」，强认知冲击，呼应 RL（L34）期望回报。  
**衔接**：L11 第一性原理（从随机到学习）；L34 PPO（奖励期望最大化）。  
**依赖**：`pip install numpy matplotlib`。

# 📚 作业 / 下一步

1. 把赢率 `0.48` 改成 `0.5`，再看平均资金曲线。
2. 搜索「大数定律」，理解为什么局数越多越接近理论值。
3. 下一课 **L11 AI 第一性原理：从猜想到学习** —— 揭开「机器如何从无到有学会预测」。